# 13 · Carry or repeat: a learned decision per tile

Every learned model that drew pixels or motion lost to the tile search, or gained a few tenths of
a dB for many times its work: two received frames do not say what happens next, and a network
unsure of it hedges. A **decision** cannot hedge. Per tile it picks one of two sharp pictures -
the search's N+1 (N moved by half the motion it found) or N itself - and a wrong pick is a
whole-tile mistake, never a blur.

There is room for one. On real screens (notebooks 11 and 12) repeat beats the search on the
moving tiles - most of them are a still picture under codec noise, which the search's gate (its
cost at no motion at least 0.1 luma levels) lets through - while the search wins where something
truly moved. Choosing per tile the better of the two is +0.50 dB over the search on the moving
real test tiles. The decision here sees only what the client has: the search's own work on the
two received frames, and the stream's bitrate (`helpers/decide.py`, `FEATURES`).

- **The control, no learning**: the search's gate raised - repeat wherever the cost at no motion is
  under a threshold, tuned on the training data.
- **Learned**: logistic regression and a small network (12 → 16 → 16 → 1) on the 12 features,
  trained to pick the picture that scores better, each tile weighted by how much better (capped
  at 50 dB, the moving tiles' score) - on the real training videos, on the drawn training clips,
  and on both; the one to test chosen on the real epoch-choosing videos.
- **The ceiling**: the better of the two per tile, which sees N+1.

Scored on notebook 12's 32 real test videos - the same split, the same stream - and on the drawn
validation set.

Steps:

1. **Data** - features and both pictures' scores, for every tile.
2. **The ceiling and the threshold** - what choosing is worth, and how much a raised gate takes.
3. **The learned decision** - trained three ways, the features it leans on.
4. **Test** - real test videos and the drawn validation set.

Needs: the YouTube sequences, `00_data_preprocess.ipynb`, and notebook 11's raw frame steps. Nothing
is exported; results cached in `checkpoints/decide/`.

In [1]:
"""Settings."""

import json
import pickle
import time

import numpy as np
import torch

from helpers import decide, models, recorded, training
from helpers.decide import FEATURES
from helpers.training import DEVICE

STILL, SHARPNESS, CAP = 0.1, 10.0, 50.0
CACHE = training.CHECKPOINT_DIR / "decide"
SEED = 0
training.seed_everything(SEED)

SCREENS = [
    "Cinema_4D/3N2nRlKlbgk", "Cinema_4D/3OnjlqkFClw", "Cinema_4D/6y8iPU4xTZs",
    "Cinema_4D/HuPwUlHWi7M", "Cinema_4D/Wb7GrS8csC8", "Cinema_4D/WqMNDw7zj24",
    "Cinema_4D/YMpf7sjLOsw", "Cinema_4D/YUF6yUFzZR8", "Cinema_4D/ek28bIHBxLE",
    "Cinema_4D/fI49YQ-c7Gg", "Cinema_4D/m077KW7jD_w", "Cinema_4D/zpNnUUQBgMo",
    "Computer/ldH3aBSYGW4", "Digital_painting/CyaXHJjh1E0", "Digital_painting/FpmtImp2jQE",
    "Digital_painting/anv74Y5n2BI", "Digital_painting/kbA1I_1LdLM", "Digital_painting/usj75M0UOIg",
    "Font/NBRptKep6vU", "Font/_pqbCCuxIeM", "Font/uEiWevVqYw0",
    "Map/VFotXCaAxbE", "Map/kOPj4JVxkaE", "Map/rSSVApBDxdk",
    "Map/tb0C1CO4ang", "Map/w1sWRlMQD3U", "Microsoft_Windows/0ElkyEf7bVU",
    "Microsoft_Windows/5XN1agV-pHg", "Microsoft_Windows/5l4GAZxhjkY", "Microsoft_Windows/65KCQsbVjao",
    "Microsoft_Windows/A5HBnSAZEtY", "Microsoft_Windows/EjpH5f9xv90", "Microsoft_Windows/OJzjrFRwouQ",
    "Microsoft_Windows/Tmd9TMyFdEM", "Microsoft_Windows/WqnDsSZJaRk", "Microsoft_Windows/ZyCTAVT_e6M",
    "Microsoft_Windows/gscBIcLm_Gw", "Microsoft_Windows/kVhdOWxQNF8", "Microsoft_Windows/muZ87U-unZE",
    "Microsoft_Windows/rqRovhzewks", "Microsoft_Windows/yAqsK_3gimg", "Microsoft_Windows/yw9xxpdfGbo",
    "Personal_computer/6IkVjaV-PxU", "Personal_computer/wMw8j9qoY24", "Screencast/1_iVs-exIEk",
    "Screencast/1cqdCQZ_rYw", "Screencast/5IWA6qNMiA8", "Screencast/BQldp1LelSY",
    "Screencast/gyogNNYHSuo", "Screencast/i_Vl80pJjdg", "Screencast/jH6J-n6zt4c",
    "Screencast/nAhUUivmD-Q", "Screencast/pykywx452N8", "Screencast/tGRfnsJnSTY",
    "Screenshot/_j1FKT3sdfw", "Sprite_computer_graphics/haAw7_HBc24", "Text_messaging/eYEFLv9do0k",
    "Web_page/AZITFc78-sA", "Web_page/GD9yAOho4zc", "Web_page/QdZj6wYAtZs",
    "Web_page/lrIYoUJqu44", "Web_page/q6CBH_TKEuE", "Website/15qRSwJ6uso",
    "Website/Hh3ms730O1o", "Website/IBA-dN_pSH8", "Website/IdZ8_S0ptJg",
    "Website/VVxYUNZ8u8U", "Website/aP-pZIQs6yE", "Website/md5YK_CxD3s",
    "Website/wCSTWjZ31iA", "Website/xNmwZ1rholY", "Computer/DyS-XDmfjmM",
    "Digital_painting/YUwbMpIbz-w", "Font/JxOaKvheZms", "Microsoft_Windows/1u954QeRdFE",
    "Microsoft_Windows/jKMgt_ugvH0", "Personal_computer/aZeEqvaJzus", "Personal_computer/zqCPKnzmcYA",
    "Sprite_computer_graphics/gmaQQmRPeeM", "Text_messaging/bkoMyKY_cAY", "Web_page/MmNPG0Gn6lc",
]


def split_by_video(videos, share, seed):
    """Per category, a seeded `share` of its videos (at least one where it has two or more) - the rest apart."""
    rng = np.random.default_rng(seed)
    chosen, rest = [], []
    for category in sorted({v.split("/")[0] for v in videos}):
        items = sorted(v for v in videos if v.startswith(category + "/"))
        rng.shuffle(items)
        count = int(round(share * len(items))) if len(items) > 1 else 0
        chosen += items[:count]
        rest += items[count:]
    return chosen, rest


TEST, REST = split_by_video(SCREENS, 31 / 81, SEED)
VAL, TRAIN = split_by_video(REST, 8 / 50, SEED + 1)

real = recorded.manifest({"train": TRAIN, "val": VAL, "test": TEST}, seed=SEED)     # notebook 12's stream
drawn = training.load_manifest()
published = models.TileSearch(4, 2, SHARPNESS, 2, 1 / real["received_every"], still=STILL).to(DEVICE).eval()
print(f"real: {len(TRAIN)} train, {len(VAL)} epoch-choosing, {len(TEST)} test videos; "
      f"drawn: {len(drawn['splits']['train']['sources'])} train, {len(drawn['splits']['val']['sources'])} val clips")

real: 42 train, 7 epoch-choosing, 32 test videos; drawn: 753 train, 115 val clips


## 1. Data

Every client tile at every received frame N from 2 on, of every clip: the 12 features, the
wPSNR of repeat and of the search's N+1 over the region the client keeps, whether the search's
gate is open (moving) and whether N+1 differs from N (changed). For the real clips also whether
N+1 is a new frame in the raw source (notebook 11, "Duplicated frames").

In [2]:
CACHE.mkdir(parents=True, exist_ok=True)


def table(name, manifest, split):
    path = CACHE / f"{name}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    started = time.time()
    found = decide.load(manifest, split, published)
    path.write_bytes(pickle.dumps(found))
    print(f"{name}: {len(found['repeat'])} tiles in {time.time() - started:.0f}s")
    return found


data = {"real train": table("real_train", real, "train"), "real val": table("real_val", real, "val"),
        "real test": table("real_test", real, "test"),
        "drawn train": table("drawn_train", drawn, "train"), "drawn val": table("drawn_val", drawn, "val")}
steps = json.loads((training.CHECKPOINT_DIR / "recorded" / "raw_steps.json").read_text())
for name, found in data.items():
    if name.startswith("real"):
        found["fresh"] = np.array([n + 1 < len(steps[f]) and steps[f][n] > 0.25 and steps[f][n + 1] > 0.25
                                   for f, n in zip(found["file"], found["frame"])])
        found["video"] = np.array([f.rsplit("/", 1)[0] for f in found["file"]])
    else:
        found["video"] = found["source"].astype(str)
    m = found["moving"]
    better = np.minimum(found["search"], CAP) > np.minimum(found["repeat"], CAP)
    print(f"{name:<12} {len(m):6} tiles, {m.mean():4.0%} moving; on the moving ones the search is better on "
          f"{better[m].mean():4.0%}, repeat on {(np.minimum(found['repeat'], CAP) > np.minimum(found['search'], CAP))[m].mean():4.0%}")

real_train: 16275 tiles in 167s
real_val: 2850 tiles in 28s
real_test: 13350 tiles in 136s
drawn_train: 80624 tiles in 681s
drawn_val: 13155 tiles in 101s
real train    16275 tiles,  53% moving; on the moving ones the search is better on  29%, repeat on  16%
real val       2850 tiles,  45% moving; on the moving ones the search is better on  27%, repeat on  14%
real test     13350 tiles,  50% moving; on the moving ones the search is better on  28%, repeat on  17%
drawn train   80624 tiles,  63% moving; on the moving ones the search is better on  59%, repeat on  17%
drawn val     13155 tiles,  56% moving; on the moving ones the search is better on  61%, repeat on  14%


## 2. The ceiling and the threshold

On the moving tiles, capped at 50 dB: the search, repeat, the better of the two per tile, and the
search's gate raised - repeat where the cost at no motion is under a threshold. The threshold is
chosen on the real training videos (and, for comparison, on the drawn training clips).

In [3]:
def chosen(found, take_search):
    """(S,) the capped wPSNR of picking the search's N+1 where `take_search`, repeat elsewhere; still tiles repeat."""
    pick = np.where(take_search & found["moving"], found["search"], found["repeat"])
    return np.minimum(pick, CAP)


def zero_cost(found):
    return np.exp(found["features"][:, 0]) - 0.01


THRESHOLDS = [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8]
print(f"{'moving tiles, ≤ 50 dB':<26}" + "".join(f"{name:>13}" for name in data))
for label, rule in (("repeat", lambda f: np.zeros(len(f["repeat"]), bool)),
                    ("search (gate 0.1)", lambda f: np.ones(len(f["repeat"]), bool)),
                    ("better of the two", lambda f: np.minimum(f["search"], CAP) >= np.minimum(f["repeat"], CAP))):
    print(f"{label:<26}" + "".join(f"{chosen(f, rule(f))[f['moving']].mean():13.2f}" for f in data.values()))
print()
for threshold in THRESHOLDS:
    print(f"{f'gate at {threshold} levels':<26}" + "".join(f"{chosen(f, zero_cost(f) >= threshold)[f['moving']].mean():13.2f}"
                                                          for f in data.values()))
GATE = {source: max(THRESHOLDS, key=lambda th: chosen(data[source], zero_cost(data[source]) >= th)[data[source]["moving"]].mean())
        for source in ("real train", "drawn train")}
print(f"\nchosen on the real training videos: {GATE['real train']} levels; on the drawn training clips: {GATE['drawn train']}")

moving tiles, ≤ 50 dB        real train     real val    real test  drawn train    drawn val
repeat                            43.91        45.84        43.91        34.61        34.05
search (gate 0.1)                 43.25        45.62        43.53        36.12        36.98
better of the two                 44.03        45.84        44.04        37.27        37.81

gate at 0.1 levels                43.25        45.62        43.53        36.12        36.98
gate at 0.25 levels               43.25        45.62        43.54        36.12        36.97
gate at 0.5 levels                43.25        45.62        43.55        36.12        36.95
gate at 0.75 levels               43.25        45.62        43.55        36.11        36.92
gate at 1 levels                  43.26        45.62        43.55        36.10        36.92
gate at 1.5 levels                43.25        45.62        43.55        36.06        36.86
gate at 2 levels                  43.24        45.62        43.56        36.04 

## 3. The learned decision

Inputs: the 12 features (`gap` as log1p), standardised on the training data. Target: whether the
search's N+1 scores better than repeat on the tile, capped at 50 dB; weight: by how much. Moving
tiles only - a still tile repeats anyway. Logistic regression and a small network, Adam over the
whole set, kept at the step that scores best on the real epoch-choosing videos (moving tiles,
capped). Trained on the real training videos, on the drawn training clips, and on both (each
source weighted to the same total).

In [4]:
def inputs(found):
    x = found["features"].copy()
    x[:, FEATURES.index("gap")] = np.log1p(x[:, FEATURES.index("gap")])
    return x


def target(found):
    gain = np.minimum(found["search"], CAP) - np.minimum(found["repeat"], CAP)
    return (gain > 0).astype(np.float32), np.abs(gain).astype(np.float32)


def fit(sources, kind, steps_max=3000, rate=1e-2):
    rows = [(inputs(data[s])[data[s]["moving"]], *(v[data[s]["moving"]] for v in target(data[s]))) for s in sources]
    x = np.concatenate([r[0] for r in rows])
    y = np.concatenate([r[1] for r in rows])
    w = np.concatenate([r[2] / r[2].sum() for r in rows])        # each source the same total weight
    mean, std = x.mean(0), x.std(0) + 1e-6
    torch.manual_seed(SEED)
    net = torch.nn.Linear(len(FEATURES), 1) if kind == "logistic" else torch.nn.Sequential(
        torch.nn.Linear(len(FEATURES), 16), torch.nn.ReLU(), torch.nn.Linear(16, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))
    xt, yt, wt = (torch.as_tensor(a, dtype=torch.float32) for a in ((x - mean) / std, y, w / w.mean()))
    optimiser = torch.optim.Adam(net.parameters(), lr=rate)
    val, val_x = data["real val"], torch.as_tensor((inputs(data["real val"]) - mean) / std, dtype=torch.float32)
    best = (-np.inf, None, 0)
    for step in range(steps_max + 1):
        if step % 25 == 0:
            with torch.no_grad():
                take = (net(val_x)[:, 0] > 0).numpy()
            score = chosen(val, take)[val["moving"]].mean()
            if score > best[0] + 1e-4:
                best = (score, {k: v.clone() for k, v in net.state_dict().items()}, step)
        loss = (torch.nn.functional.binary_cross_entropy_with_logits(net(xt)[:, 0], yt, reduction="none") * wt).mean()
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    net.load_state_dict(best[1])

    def decide_tiles(found):
        with torch.no_grad():
            return (net(torch.as_tensor((inputs(found) - mean) / std, dtype=torch.float32))[:, 0] > 0).numpy()
    return decide_tiles, net, best


TRAINED = {}
for sources in (("real train",), ("drawn train",), ("real train", "drawn train")):
    for kind in ("logistic", "network"):
        name = f"{kind}, {' + '.join(s.split()[0] for s in sources)}"
        TRAINED[name] = fit(sources, kind)
        print(f"{name:<28} real epoch-choosing videos: {TRAINED[name][2][0]:.3f} dB (step {TRAINED[name][2][2]})")
val = data["real val"]
print(f"{'(the search)':<28} real epoch-choosing videos: {chosen(val, np.ones(len(val['repeat']), bool))[val['moving']].mean():.3f} dB")
BEST = max(TRAINED, key=lambda name: TRAINED[name][2][0])
print(f"\nchosen: {BEST}")
weights = TRAINED["logistic, real + drawn"][1].weight.detach().numpy()[0]
print("\nlogistic regression on both, standardised weights (positive: carry the motion):")
for name, value in sorted(zip(FEATURES, weights), key=lambda p: -abs(p[1])):
    print(f"  {name:<20}{value:+7.2f}")

logistic, real               real epoch-choosing videos: 45.842 dB (step 925)
network, real                real epoch-choosing videos: 45.843 dB (step 125)
logistic, drawn              real epoch-choosing videos: 45.837 dB (step 175)
network, drawn               real epoch-choosing videos: 45.842 dB (step 0)
logistic, real + drawn       real epoch-choosing videos: 45.842 dB (step 25)
network, real + drawn        real epoch-choosing videos: 45.843 dB (step 175)
(the search)                 real epoch-choosing videos: 45.616 dB

chosen: network, real

logistic regression on both, standardised weights (positive: carry the motion):
  explained             -0.48
  log bpp               -0.29
  deblock               -0.26
  residual share        -0.24
  changed pixels        -0.19
  log zero cost         -0.17
  change strength       -0.11
  gap                   +0.10
  log motion            -0.09
  log best cost         -0.05
  unexplained cells     +0.01
  peak                  -0.00


## 4. Test

The real test videos - never trained on nor used to choose - and the drawn validation set. On the
moving tiles (capped) with the gain over the search and its 95% range over videos (clips for the
drawn set); on the changed tiles; on the real changed tiles where N+1 is a new frame; on all tiles.

In [5]:
rng = np.random.default_rng(SEED)


def over(gain, mask, groups, draws=2000):
    per = [gain[mask & (groups == g)] for g in np.unique(groups[mask])]
    means = [np.concatenate([per[i] for i in rng.integers(0, len(per), len(per))]).mean() for _ in range(draws)]
    return np.percentile(means, [2.5, 97.5])


RULES = {"repeat": lambda f: np.zeros(len(f["repeat"]), bool),
         "search (gate 0.1)": lambda f: np.ones(len(f["repeat"]), bool),
         f"gate raised to {GATE['real train']} (real)": lambda f: zero_cost(f) >= GATE["real train"],
         f"gate raised to {GATE['drawn train']} (drawn)": lambda f: zero_cost(f) >= GATE["drawn train"],
         **{f"learned: {name}": TRAINED[name][0] for name in TRAINED},
         "better of the two (sees N+1)": lambda f: np.minimum(f["search"], CAP) >= np.minimum(f["repeat"], CAP)}
for name in ("real test", "drawn val"):
    found = data[name]
    moving, changed = found["moving"], found["changed"]
    base = chosen(found, np.ones(len(moving), bool))
    print(f"\n{'=' * 40}\n{name}: {len(moving)} tiles, {moving.sum()} moving, {changed.sum()} changed"
          + (f", {(changed & found['fresh']).sum()} changed with N+1 new" if "fresh" in found else "") + f"\n{'=' * 40}")
    head = f"{'':<36}{'moving, ≤ 50 dB':>17}{'95% over videos':>18}{'changed':>10}"
    print(head + (f"{'changed, N+1 new':>18}" if "fresh" in found else "") + f"{'all':>9}{'tiles to search':>17}")
    for rule_name, rule in RULES.items():
        take = rule(found) & moving
        score = chosen(found, take)
        uncapped = np.where(take, found["search"], found["repeat"])
        gain = score - base
        low, high = over(gain, moving, found["video"]) if rule_name != "search (gate 0.1)" else (0, 0)
        line = (f"{rule_name:<36}{score[moving].mean():9.2f} ({gain[moving].mean():+.2f})  [{low:+.2f}, {high:+.2f}]"
                f"{uncapped[changed].mean():10.2f}")
        if "fresh" in found:
            line += f"{uncapped[changed & found['fresh']].mean():18.2f}"
        print(line + f"{uncapped.mean():9.2f}{take[moving].mean():16.0%}")


real test: 13350 tiles, 6732 moving, 2503 changed, 1247 changed with N+1 new
                                      moving, ≤ 50 dB   95% over videos   changed  changed, N+1 new      all  tiles to search
repeat                                  43.91 (+0.37)  [+0.17, +0.62]     33.54             30.60    62.79              0%
search (gate 0.1)                       43.53 (+0.00)  [+0.00, +0.00]     33.47             30.81    62.47            100%
gate raised to 8 (real)                 43.58 (+0.05)  [-0.02, +0.12]     33.39             30.65    62.55             11%
gate raised to 0.1 (drawn)              43.53 (+0.00)  [+0.00, +0.00]     33.47             30.81    62.47            100%
learned: logistic, real                 43.91 (+0.38)  [+0.18, +0.63]     33.60             30.74    62.79             53%
learned: network, real                  43.90 (+0.37)  [+0.17, +0.61]     33.59             30.74    62.77             82%
learned: logistic, drawn                43.95 (+0.41)  [+0

## What it says

- **On real screens, the decision learns to repeat.** Every learned rule ends where repeat already
  is: +0.37 to +0.41 dB over the search on the moving test tiles (repeat itself +0.37, 95% over
  videos +0.17 to +0.62). The ceiling - the better picture per tile, seeing N+1 - is +0.50: repeat
  takes three quarters of it with no decision at all, and the best learned rule (logistic,
  trained on the drawn clips) adds +0.04 of the remaining +0.13. On the changed tiles where N+1 is
  new it is 30.93 dB against the search's 30.81 and the ceiling's 31.28.
- **A raised gate is not the answer either.** Repeat wherever the search's cost at no motion is
  under a threshold: from 0.1 up to 8 luma levels, +0.05 dB at most on the real test tiles. The
  tiles where repeat beats the search are not quiet ones under codec noise; something changed
  there that one motion gets wrong.
- **What the decision leans on**: how little of the change the best motion explains (`explained`,
  by far), then the bitrate and loop filter, the residual after aligning N−2, and how many pixels
  changed. It reads "does one motion account for this tile?" - the question the search already
  asks - and answers "repeat" more readily.
- **Decisions do not travel between the two kinds of picture.** Trained on real screens, they lose
  1.8 to 2.2 dB on the drawn validation set by repeating its scrolls; trained on the drawn clips,
  the logistic one is the search there (+0.01) and slightly better than repeat on real screens.
  The small network trained on both is closest to the better fixed choice on each: +0.38 on the
  real test tiles (repeat's level) and −0.07 on the drawn ones (the search's level) - a switch by
  kind of content, not a better pick per tile. (The network trained on the drawn clips alone kept
  its untrained step: chosen on real videos, the step that repeats everything scored best.)
- **Per tile, two frames do not say which picture will be right.** On the drawn set the ceiling is
  +0.83 dB over the search, and no rule found any of it; on real screens, +0.13 over repeat, and
  the best found +0.04. The same wall as every learned model before: what decides between the
  two pictures is what happens between N and N+1.

So a learned decision is worth having only as a **switch by content**: repeat where pictures change
the way tutorials do, carry the motion where they scroll and pan. That is cheap - 12 numbers the
search mostly computes already and a 500-weight network - and never far below the better of the
two fixed choices on either set. It does not beat them. On these recordings, extrapolation's best
showing is repeat; whether that holds for a live 60 fps desktop is the question only a live session
answers.